# 📈 Regression Workflow Template
### A complete, runnable scaffold for supervised regression projects

---

## What is regression?

Regression is a type of supervised machine learning where the goal is to predict a **continuous numeric value** — for example:
- Predicting house prices
- Forecasting sales revenue
- Estimating patient age from medical data

This is different from **classification**, where you predict a category (yes/no, spam/not spam). Here, the output is a real number.

---

## How to use this notebook

1. **Search for `# ← UPDATE`** — those are the only lines you need to change for your own dataset
2. **Run cells top-to-bottom** — each step builds on the previous one
3. **Read the markdown cells** before each code block — they explain *what* is happening and *why*

---

## The full workflow at a glance

```
Step 1  →  Imports & Configuration
Step 2  →  Load Data
Step 3  →  Feature Engineering
Step 4  →  Exploratory Data Analysis (EDA)
Step 5  →  Train / Test Split
Step 6  →  Preprocessing Pipeline
Step 7  →  Helper Functions (plotting)
Step 8  →  Model Zoo + 5-Fold Cross-Validation
Step 9  →  Hyperparameter Tuning
Step 10 →  Final Test Evaluation
Step 11 →  Feature Importance
Step 12 →  Per-Model Scatter + Residual Plots
Step 13 →  Combined Predicted vs Actual (all models)
Step 14 →  SHAP Interpretability
Step 15 →  Conclusion & Checklist
```

---
## Step 1 — Imports & Configuration

We import everything we need up front. This makes it easy to see all dependencies at a glance and avoids confusing errors later in the notebook.

**Key libraries:**
- `numpy` / `pandas` — the backbone of data work in Python (arrays and tables)
- `matplotlib` / `seaborn` — plotting and visualisation
- `sklearn` — scikit-learn: the go-to library for machine learning in Python
- `xgboost` / `lightgbm` — fast, powerful gradient boosting models

**Configuration set here:**
- `RANDOM_STATE = 123` — a fixed seed so results are reproducible (same number = same output every run)
- Colour palette constants — used to keep all plots visually consistent

In [ ]:
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Model selection utilities ──────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,   # splits data into train and test sets
    KFold,              # k-fold cross-validation (non-stratified, used for regression)
    cross_validate,     # runs CV and returns multiple metrics at once
    GridSearchCV,       # exhaustive search over a grid of hyperparameters
)

# ── Preprocessing ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler,         # normalises numeric features to mean=0, std=1
    OneHotEncoder,          # converts categorical text columns into binary (0/1) columns
    FunctionTransformer,    # wraps a custom Python function as a sklearn transformer
)
from sklearn.impute import SimpleImputer          # fills missing (NaN) values
from sklearn.feature_selection import VarianceThreshold  # removes constant/useless columns

# ── Pipeline utilities ─────────────────────────────────────────────────────────
from sklearn.pipeline import Pipeline             # chains preprocessing + model into one object
from sklearn.compose import ColumnTransformer     # applies different transforms to different columns

# ── Regression metrics ─────────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_absolute_error,           # MAE  — average absolute error (same units as target)
    mean_squared_error,            # MSE  — penalises large errors more than MAE
    r2_score,                      # R²   — proportion of variance explained (1.0 = perfect)
    mean_absolute_percentage_error,# MAPE — error as a % of the actual value
)

# ── Regression models ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ── Global config ─────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore")   # suppress noisy sklearn / lightgbm warnings
RANDOM_STATE = 123                  # fixed seed — change to any integer, keep it constant

# Shared colour palette for all plots
CLR_DARK  = "#2c3e50"   # dark navy
CLR_BLUE  = "#3498db"   # scatter points
CLR_GREEN = "#2ecc71"   # residual points
CLR_RED   = "#e74c3c"   # error / highlight

print("✅ Imports and configuration ready.")

---
## Step 2 — Load Data

Replace the dummy dataset block with your own file.

**After loading, always run these inspection steps:**
- `df.shape` — how many rows and columns?
- `df.dtypes` — are numeric columns typed as numbers? Text as `object`?
- `df.head()` — do the values look sensible?
- `df.describe()` — are ranges and means realistic?
- Check your target column: what are its min, max, mean, and distribution?

> 📌 **Unit of analysis:** What does one row represent — a property? A customer? A time period?  
> This matters for interpreting predictions correctly.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ← UPDATE: Replace the block below with your real data.
# Example:  df = pd.read_csv("your_dataset.csv")
# ─────────────────────────────────────────────────────────────────────────────

# Dummy dataset — delete from here ...
np.random.seed(RANDOM_STATE)
n_samples = 1000
df = pd.DataFrame({
    "feature1":          np.random.randn(n_samples),
    "feature2":          np.random.rand(n_samples) * 100,
    "feature3":          np.random.randint(0, 50, n_samples),
    "category1":         np.random.choice(["A", "B", "C", "Rare"], n_samples, p=[0.4, 0.4, 0.18, 0.02]),
    "category2":         np.random.choice(["X", "Y"], n_samples),
    "target_continuous": 50 + 10 * np.random.randn(n_samples) + 0.5 * np.random.rand(n_samples) * 100,
})
# ... to here once you have your own data.

# ─────────────────────────────────────────────────────────────────────────────
# Initial inspection — always run these after loading
# ─────────────────────────────────────────────────────────────────────────────
print("Dataset shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nSummary statistics:")
display(df.describe())

---
## Step 3 — Feature Engineering

Feature engineering means creating new, more informative columns from the raw data *before* modelling.
Good features often matter more than choosing the "best" model.

**Common transformations for regression:**

| Transformation | When to use | Example |
|----------------|-------------|--------|
| **Log transform** | Right-skewed targets or features (income, price) | `np.log1p(price)` |
| **Ratio features** | Raw totals that depend on group size | `rooms / households` |
| **Polynomial terms** | Non-linear relationships | `area²`, `age × income` |
| **Date decomposition** | Datetime columns | `month`, `days_since_launch` |
| **Binning** | Continuous → grouped categories | `pd.cut(age, bins=[0,18,65,99])` |

> ⚠️ **Log-transforming the target:** If your target is skewed (e.g. house prices), log-transforming it often improves model performance. Remember to `np.expm1()` your predictions at the end to get back to the original scale.

In [ ]:
# ← UPDATE: Add your feature engineering here if needed.
# Examples:
# df['rooms_per_household'] = df['total_rooms'] / df['households']
# df['log_income']          = np.log1p(df['median_income'])
# df['log_target']          = np.log1p(df['price'])   # then use 'log_target' as TARGET_COL below

# If using the dummy data, nothing to do here.
print("✅ Feature engineering complete. Shape:", df.shape)

---
## Step 4 — Exploratory Data Analysis (EDA)

**Never fit a model on data you haven't understood.**

EDA answers four key questions before modelling:

1. **Target distribution** — is it roughly normal, or heavily skewed?
2. **Missing values** — which columns have gaps, and how many?
3. **Feature distributions** — any extreme outliers? Constant columns?
4. **Correlations** — which features are most related to the target? Any nearly-identical features?

> 📊 For regression, R² is your headline metric, but also track MAE (in the same units as your target) for intuitive interpretation.

In [ ]:
# ← UPDATE: Set your actual target column name here
TARGET_COL = "target_continuous"

# Identify numeric and categorical feature columns (excluding the target)
numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET_COL).tolist()
cat_cols     = df.select_dtypes(include="object").columns.tolist()

print(f"Numeric features   ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

# ── 4.1  Target distribution ──────────────────────────────────────────────────
# A roughly bell-shaped target is easiest for linear models.
# A right-skewed target (long right tail) may benefit from log-transformation.
print("\n" + "=" * 50)
print("4.1  Target distribution")
print("=" * 50)
print(df[TARGET_COL].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET_COL].hist(bins=40, ax=axes[0], color=CLR_BLUE, edgecolor='white')
axes[0].set_title("Target Distribution (histogram)"); axes[0].set_xlabel(TARGET_COL)
axes[0].axvline(df[TARGET_COL].mean(), color=CLR_RED, lw=2, linestyle='--', label='mean')
axes[0].legend()

# Box plot — quickly shows outliers (dots beyond the whiskers)
axes[1].boxplot(df[TARGET_COL], vert=False, patch_artist=True,
                boxprops=dict(facecolor=CLR_BLUE, alpha=0.7))
axes[1].set_title("Target Distribution (box plot)"); axes[1].set_xlabel(TARGET_COL)
plt.tight_layout(); plt.show()

# ── 4.2  Missing values ───────────────────────────────────────────────────────
# The pipeline will handle these automatically, but good to know which columns are affected.
print("\n" + "=" * 50)
print("4.2  Missing values (% per column)")
print("=" * 50)
missing = (df.isnull().mean() * 100).round(2)
missing = missing[missing > 0]
print(missing if len(missing) > 0 else "No missing values found 🎉")

# ── 4.3  Feature distributions ───────────────────────────────────────────────
# Check for skewed or unusual distributions in numeric features.
print("\n" + "=" * 50)
print("4.3  Numeric feature distributions")
print("=" * 50)
if numeric_cols:
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5 * len(numeric_cols), 4))
    if len(numeric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, numeric_cols):
        df[col].hist(ax=ax, bins=30, color=CLR_BLUE, edgecolor='white')
        ax.set_title(col)
    plt.suptitle("Numeric Feature Distributions", fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

# ── 4.4  Correlation with target ─────────────────────────────────────────────
# Pearson correlation measures linear association (range: -1 to +1).
# High absolute value = strong linear relationship with the target.
print("\n" + "=" * 50)
print("4.4  Correlation matrix (numeric features + target)")
print("=" * 50)
if len(numeric_cols) >= 1:
    corr = df[numeric_cols + [TARGET_COL]].corr()
    plt.figure(figsize=(max(6, len(numeric_cols) + 2), max(5, len(numeric_cols) + 1)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, linewidths=0.5)
    plt.title("Pearson Correlation Matrix", fontweight='bold')
    plt.tight_layout(); plt.show()

    # Print target correlations in plain text for quick reading
    target_corr = corr[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
    print("\nFeature correlations with target (sorted by absolute value):")
    print(target_corr.round(3))

---
## Step 5 — Train / Test Split

We divide the data into two parts:
- **Training set (80%)** — the model learns from this
- **Test set (20%)** — held completely aside; only used at the very end to measure real-world performance

**Critical rule — split BEFORE any preprocessing:**
```
Split first  →  fit preprocessor on train only  →  transform train
                                                  →  transform test (using train stats)
```
If you scale or impute *before* splitting, your training data "sees" test statistics.
This is **data leakage** — it inflates your performance metrics and won't hold up in production.

> Note: Unlike classification, we do **not** use `stratify=y` for regression because `stratify` requires discrete categories. The random split is sufficient here.

In [ ]:
# Separate features (X) from the target (y)
X = df.drop(columns=[TARGET_COL])   # all columns except the target
y = df[TARGET_COL]                   # the continuous target column

# Re-detect feature types on X alone (no target contamination)
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols     = X.select_dtypes(include="object").columns.tolist()

# Split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,              # 20% held out for final evaluation
    random_state=RANDOM_STATE,   # reproducible split
    # No stratify= here — regression targets are continuous
)

print(f"Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows")
print(f"\nTarget — Train: mean={y_train.mean():.2f}, std={y_train.std():.2f}")
print(f"Target — Test:  mean={y_test.mean():.2f},  std={y_test.std():.2f}")

---
## Step 6 — Preprocessing Pipeline

Raw data is almost never ready for a model. We build preprocessing **Pipelines** that handle:

**Numeric features:**
1. `SimpleImputer(strategy='median')` — fills missing values with the column median (robust to outliers)
2. `VarianceThreshold(threshold=0)` — drops any column with zero variance (a constant column has no predictive power)
3. `StandardScaler()` — normalises to mean=0, std=1 (essential for Ridge, Lasso, and KNN; harmless for trees)

**Categorical features:**
1. `SimpleImputer(strategy='most_frequent')` — fills missing values with the most common value
2. Rare-category collapser — merges categories appearing in < 3% of rows into `'Other'` (avoids sparse, noisy dummies)
3. `OneHotEncoder` — converts text categories into binary (0/1) columns
   - `handle_unknown='ignore'` — any new category seen at test time becomes all-zeros (safe fallback)
   - `drop='first'` — removes one dummy per variable to avoid perfect multicollinearity

**Why Pipeline instead of manual transforms?**
- ✅ Prevents leakage — the scaler and imputer are re-fitted on each training fold during CV
- ✅ One-line `.fit()` / `.predict()` at deployment
- ✅ The entire object (preprocessor + model) can be saved to disk with `joblib.dump()`

In [ ]:
# ── Rare-category collapser ───────────────────────────────────────────────────
# Categories that appear in fewer than 3% of rows are merged into 'Other'.
# Without this, OneHotEncoder creates many near-empty columns that add noise.
def collapse_rare_categories(X_df, threshold=0.03):
    """Replace categories with frequency < threshold → 'Other'."""
    out = X_df.copy()
    for col in out.columns:
        counts = out[col].value_counts(normalize=True)
        rare   = counts[counts < threshold].index
        out[col] = out[col].apply(lambda x: "Other" if x in rare else x)
    return out

# Wrap as a sklearn-compatible transformer
rare_collapser = FunctionTransformer(
    lambda X: collapse_rare_categories(pd.DataFrame(X, columns=cat_cols)),
    validate=False,
)

# ── Numeric sub-pipeline ──────────────────────────────────────────────────────
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # fill NaN with median
    ("zv",     VarianceThreshold(threshold=0)),     # drop zero-variance columns
    ("scale",  StandardScaler()),                   # normalise
])

# ── Categorical sub-pipeline ──────────────────────────────────────────────────
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),  # fill NaN with mode
    ("rare",   rare_collapser),                          # collapse rare values
    ("ohe",    OneHotEncoder(
                   handle_unknown="ignore",  # unknown test categories → all zeros
                   sparse_output=False,      # return dense array (easier to work with)
                   drop="first",             # drop one dummy per feature to avoid multicollinearity
              )),
])

# ── Combined ColumnTransformer ────────────────────────────────────────────────
# Applies num_pipe to numeric columns and cat_pipe to categorical columns in parallel,
# then concatenates the results into a single feature matrix.
preprocessor = ColumnTransformer([
    ("num", num_pipe, numeric_cols),
    ("cat", cat_pipe, cat_cols),
])

print("✅ Preprocessing pipeline ready.")

---
## Step 7 — Helper Functions (Plotting)

We define reusable plot functions once here, then call them for every model.  
Consistent visuals make it easy to compare models side-by-side.

**Two core regression diagnostic plots:**

| Plot | What to look for |
|------|------------------|
| **Predicted vs Actual** | Points should hug the diagonal line. Deviations show where the model over/under-predicts. |
| **Residual plot** | Residuals (= actual − predicted) should be randomly scattered around zero. Patterns indicate the model is missing something systematic. |

In [ ]:
def plot_regression_scatter(ax, y_true, y_pred, title):
    """
    Predicted vs Actual scatter plot.

    The dashed diagonal = perfect prediction.
    Points above the line = model over-predicted.
    Points below the line = model under-predicted.
    Metrics shown: R², MAE, RMSE.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    # Draw the "perfect prediction" reference line
    lo, hi = min(np.min(y_true), np.min(y_pred)), max(np.max(y_true), np.max(y_pred))
    ax.plot([lo, hi], [lo, hi], "k--", lw=2, label="Perfect Prediction")

    ax.scatter(y_true, y_pred, alpha=0.6, s=30, color=CLR_BLUE)
    ax.set_xlabel("Actual Values")
    ax.set_ylabel("Predicted Values")
    ax.set_title(title)

    # Overlay key metrics in a text box
    ax.text(0.05, 0.95, f"R²={r2:.3f}\nMAE={mae:.3f}\nRMSE={rmse:.3f}",
            transform=ax.transAxes, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    ax.legend(); ax.grid(True, alpha=0.3)


def plot_residuals(ax, y_true, y_pred, title):
    """
    Residuals vs Predicted values.

    Residual = actual − predicted.
    The red horizontal line is the zero-error baseline.
    A good model: residuals randomly scattered around zero with no clear pattern.
    Red flag: a funnel shape (heteroscedasticity) means errors grow with the prediction size.
    """
    residuals = np.array(y_true) - np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    ax.scatter(y_pred, residuals, alpha=0.6, s=30, color=CLR_GREEN)
    ax.axhline(y=0, color="r", linestyle="-", lw=2)  # zero-error reference
    ax.set_xlabel("Predicted Values")
    ax.set_ylabel("Residuals (Actual − Predicted)")
    ax.set_title(title)
    ax.text(0.05, 0.95, f"MAE={mae:.3f}",
            transform=ax.transAxes, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    ax.grid(True, alpha=0.3)


def plot_feature_importance(pipeline, top_n=15):
    """
    Horizontal bar chart of the top N feature importances.

    Works for:
    - Tree models (RandomForest, XGBoost, LightGBM) — uses .feature_importances_
    - Linear models (Ridge, Lasso, LinearRegression) — uses absolute .coef_ values
    """
    pre = pipeline.named_steps["pre"]
    reg = pipeline.named_steps["reg"]

    # Recover feature names after all transformations (including OHE expansion)
    try:
        feature_names = pre.get_feature_names_out()
    except AttributeError:
        feature_names = [f"Feature_{i}" for i in range(X_train.shape[1])]

    if hasattr(reg, "feature_importances_"):
        importances  = reg.feature_importances_
        metric_label = "Feature Importance"
    elif hasattr(reg, "coef_"):
        importances  = np.abs(reg.coef_)   # absolute value — size matters, not sign
        metric_label = "Absolute Coefficient"
    else:
        print(f"⚠️  Feature importance not supported for {reg.__class__.__name__}")
        return

    # Build tidy DataFrame; strip pipeline prefixes (e.g. 'num__feature1' → 'feature1')
    imp_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
    imp_df["Feature"] = imp_df["Feature"].str.split("__").str[-1]
    imp_df = imp_df.sort_values("Importance", ascending=True).tail(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(imp_df["Feature"], imp_df["Importance"], color=CLR_BLUE)
    plt.title(f"Top {top_n} {metric_label}s — {reg.__class__.__name__}", fontweight="bold")
    plt.xlabel(metric_label)
    plt.tight_layout()
    plt.show()


print("✅ Helper functions ready: plot_regression_scatter, plot_residuals, plot_feature_importance")

---
## Step 8 — Model Zoo + 5-Fold Cross-Validation

### 8.1 Why compare multiple models?

No single algorithm is best for every dataset. We run all of them with default settings first,
then invest tuning time only on the winners.

| Algorithm | Family | Needs scaling? | Notes |
|-----------|--------|---------------|-------|
| Linear Regression | Linear | ✅ Yes | Interpretable baseline; assumes linear relationships |
| **Ridge** | Linear (L2 regularised) | ✅ Yes | Like LR but handles correlated features well |
| Lasso | Linear (L1 regularised) | ✅ Yes | Can zero-out weak features (built-in feature selection) |
| Decision Tree | Tree | ❌ No | Easy to visualise but overfits |
| **Random Forest** | Ensemble (Bagging) | ❌ No | Robust; rarely overfits badly |
| KNN | Instance-based | ✅ Yes | Predicts from nearest neighbours; slow on large data |
| **XGBoost** | Ensemble (Boosting) | ❌ No | Often top performer; powerful but slower |
| **LightGBM** | Ensemble (Boosting) | ❌ No | Faster than XGBoost on large datasets |

### 8.2 Why 5-fold cross-validation?

Instead of a single train/validate split (one noisy estimate), 5-fold CV:
- Trains and validates 5 times, each time on a different 80/20 split of the training data
- Gives a much more reliable, stable average performance
- Also reports standard deviation — high std = model is sensitive to which data it sees

**Metrics used:**
- **R²** — how much of the variance in the target does the model explain? (1.0 = perfect, 0 = same as predicting the mean)
- **MAE** — mean absolute error in the *same units* as the target (e.g. dollars, kg)
- **RMSE** — like MAE but penalises large errors more heavily

In [ ]:
# ── Define the model zoo ──────────────────────────────────────────────────────
# Note: for regression we do NOT use LabelEncoder (no integer target encoding needed)
# Note: for regression we use KFold (not StratifiedKFold — that's for classification)
models = {
    "Linear Regression": LinearRegression(),
    "Ridge":             Ridge(random_state=RANDOM_STATE),        # L2 regularisation
    "Lasso":             Lasso(random_state=RANDOM_STATE),        # L1 regularisation
    "Decision Tree":     DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest":     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "KNN":               KNeighborsRegressor(n_neighbors=5),
    "XGBoost":           XGBRegressor(random_state=RANDOM_STATE),
    "LightGBM":          LGBMRegressor(random_state=RANDOM_STATE, verbose=-1),
}
print(f"✅ {len(models)} baseline models defined.")

# ── 5-fold cross-validation ───────────────────────────────────────────────────
# KFold (not StratifiedKFold) — regression targets are continuous, not discrete classes
cv5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# We ask for three metrics at once
scoring = {
    "r2":       "r2",
    "neg_mae":  "neg_mean_absolute_error",   # sklearn always negates 'loss' metrics
    "neg_rmse": "neg_root_mean_squared_error",
}

cv_results = []
print("\nRunning 5-fold CV for each model (may take ~30–60 seconds)...")

for name, reg in models.items():
    # Wrap preprocessor + model into one Pipeline to prevent leakage during CV
    pipe = Pipeline([("pre", preprocessor), ("reg", reg)])

    cv_scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv5,
        scoring=scoring,
        n_jobs=-1,   # use all CPU cores
    )

    cv_results.append({
        "Model":     name,
        "Mean R²":   cv_scores["test_r2"].mean(),
        "Std R²":    cv_scores["test_r2"].std(),
        "Mean MAE":  -cv_scores["test_neg_mae"].mean(),   # negate back to positive
        "Mean RMSE": -cv_scores["test_neg_rmse"].mean(),
    })

results_df = pd.DataFrame(cv_results).sort_values("Mean R²", ascending=False)

print("\n" + "=" * 60)
print("CV RESULTS (BASELINE, 5-FOLD, SORTED BY R²)")
print("=" * 60)
print(results_df.round(4).to_string(index=False))

# Bar chart of mean R² with error bars (standard deviation across folds)
fig, ax = plt.subplots(figsize=(10, 5))
colors = [CLR_GREEN if i == 0 else CLR_BLUE for i in range(len(results_df))]
ax.barh(results_df["Model"], results_df["Mean R²"],
        xerr=results_df["Std R²"], align="center",
        color=colors, alpha=0.85, capsize=5)
ax.set_xlabel("Mean R² (5-fold CV)", fontsize=11)
ax.set_title("Baseline Model Comparison — 5-Fold CV", fontweight="bold")
ax.axvline(0, linestyle="--", color="gray", alpha=0.6)
plt.tight_layout(); plt.show()

---
## Step 9 — Hyperparameter Tuning

After identifying the best-performing models from CV, we search for better hyperparameters.

**Key hyperparameters for regression:**

| Parameter | Model | What it controls |
|-----------|-------|------------------|
| `alpha` | Ridge / Lasso | Regularisation strength — higher = heavier penalty on large coefficients |
| `n_estimators` | RF / XGBoost | Number of trees — more = slower but smoother predictions |
| `max_depth` | RF / XGBoost | Maximum tree depth — shallower = less risk of overfitting |
| `learning_rate` | XGBoost | Step size per tree — smaller = needs more trees but often better |
| `min_samples_split` | Random Forest | Minimum samples to split a node — higher = more regularised |

> ⚠️ Always tune with `scoring='r2'`, not `'neg_mean_squared_error'`, unless MAE minimisation is your actual business goal.

In [ ]:
# ← UPDATE: Adjust the models and parameter grids to match your CV leaderboard.
# Rule of thumb: tune the top 2–3 models from the CV results above.
tuning_configs = {
    "Ridge": {
        "model": Ridge(random_state=RANDOM_STATE),
        "params": {
            "reg__alpha": [0.1, 1.0, 10.0, 100.0],  # higher alpha = stronger regularisation
        },
    },
    "RandomForest": {
        "model": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "reg__n_estimators":      [100, 200],       # more trees = more stable
            "reg__max_depth":         [5, 10, None],    # None = grow until leaves are pure
            "reg__min_samples_split": [2, 5],           # higher = more regularised
        },
    },
    "XGBoost": {
        "model": XGBRegressor(random_state=RANDOM_STATE),
        "params": {
            "reg__n_estimators":  [100, 200],
            "reg__learning_rate": [0.05, 0.1],    # smaller = more trees needed but often better
            "reg__max_depth":     [3, 5],          # shallower usually reduces overfitting
        },
    },
}

tuning_results = []
grid_objects   = {}   # store GridSearchCV objects so we can retrieve best estimators later

print("Starting Grid Search...")
print("=" * 55)

for name, config in tuning_configs.items():
    tuning_pipeline = Pipeline([("pre", preprocessor), ("reg", config["model"])])

    grid_search = GridSearchCV(
        tuning_pipeline,
        config["params"],
        cv=cv5,
        scoring="r2",    # ← optimise for R²; change to 'neg_mean_absolute_error' if preferred
        n_jobs=-1,
        verbose=0,
    )

    start_t = time.time()
    grid_search.fit(X_train, y_train)
    elapsed = time.time() - start_t

    grid_objects[name] = grid_search
    tuning_results.append({
        "Model":    name,
        "Best R²":  grid_search.best_score_,
        "Time (s)": round(elapsed, 1),
    })
    print(f"[✓] {name:<22} {elapsed:>5.1f}s  |  Best CV R²: {grid_search.best_score_:.4f}")
    print(f"    Best params: {grid_search.best_params_}")

# Pick the model with the highest tuned R²
best_model_name = sorted(tuning_results, key=lambda x: x["Best R²"], reverse=True)[0]["Model"]
best_pipeline   = grid_objects[best_model_name].best_estimator_   # already fitted

print(f"\n🏆 Best model: {best_model_name}")

---
## Step 10 — Final Evaluation on the Test Set

**This is the moment of truth.**

Everything until now (CV, tuning) was done entirely on training data.  
The test set has never been touched. We evaluate the best tuned model on it now and report four metrics:

| Metric | Interpretation | Best value |
|--------|----------------|------------|
| **R²** | % of target variance explained | 1.0 |
| **MAE** | Average absolute error (same unit as target) | 0 |
| **RMSE** | Like MAE but penalises large errors more | 0 |
| **MAPE** | Error as a percentage of the actual value | 0% |

> 💡 A CV R² of 0.82 and a test R² of 0.80 is healthy — a small drop is expected.  
> A large drop (> 0.10) suggests overfitting — try more regularisation or a simpler model.

In [ ]:
print("=" * 55)
print(f"TEST SET RESULTS — {best_model_name}")
print("=" * 55)

# Generate predictions on the untouched test set
y_pred = best_pipeline.predict(X_test)

# Compute all four metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"R²:    {r2:.4f}   (1.0 = perfect)")
print(f"MAE:   {mae:.4f}  (same units as target)")
print(f"RMSE:  {rmse:.4f}  (penalises large errors)")
print(f"MAPE:  {mape:.2%}  (% error relative to actual)")

# Side-by-side: Predicted vs Actual + Residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_regression_scatter(
    ax=axes[0],
    y_true=y_test,
    y_pred=y_pred,
    title=f"{best_model_name} — Predicted vs Actual (Test Set)",
)

plot_residuals(
    ax=axes[1],
    y_true=y_test,
    y_pred=y_pred,
    title=f"{best_model_name} — Residuals (Test Set)",
)

plt.tight_layout(); plt.show()

---
## Step 11 — Feature Importance

Feature importance tells us **which inputs the model relied on most** when making predictions.

- **Tree models** (Random Forest, XGBoost, LightGBM): uses `.feature_importances_` — how much each feature reduced prediction error across all trees
- **Linear models** (Ridge, Lasso, Linear Regression): uses `abs(coef_)` — how much a one-unit change in each feature shifts the prediction

> ⚠️ Importance tells you *which* features matter, not *how* they affect the target. Use SHAP (Step 14) for that.

In [ ]:
# Plot top 10 features for the best tuned model
plot_feature_importance(best_pipeline, top_n=10)

---
## Step 12 — Per-Model Scatter & Residual Plots

We now fit all baseline models on the full training set and plot two diagnostics for each:
- **Left:** Predicted vs Actual — how close are predictions to the truth?
- **Right:** Residuals — are errors random, or does the model systematically fail in certain ranges?

This gives a rich side-by-side comparison of every model's behaviour — not just a single number.

In [ ]:
print("Fitting all models and generating diagnostic plots...")

num_models = len(models)
fig, axes  = plt.subplots(nrows=num_models, ncols=2, figsize=(14, 5 * num_models))

if num_models == 1:
    axes = [axes]

for i, (name, reg) in enumerate(models.items()):
    # Build a fresh pipeline for this model and train it
    pipe = Pipeline([("pre", preprocessor), ("reg", reg)])
    pipe.fit(X_train, y_train)

    y_pred_i = pipe.predict(X_test)

    # ── Left: Predicted vs Actual ─────────────────────────────────────────────
    plot_regression_scatter(
        ax=axes[i][0],
        y_true=y_test,
        y_pred=y_pred_i,
        title=f"{name} — Predicted vs Actual",
    )

    # ── Right: Residuals ──────────────────────────────────────────────────────
    plot_residuals(
        ax=axes[i][1],
        y_true=y_test,
        y_pred=y_pred_i,
        title=f"{name} — Residuals",
    )

plt.tight_layout(pad=3.0)
plt.show()

---
## Step 13 — Combined Predicted vs Actual (All Models)

Overlaying all models on one chart makes it easy to see which model's predictions cluster
most tightly around the perfect-prediction line — and where each model struggles.

Points that fall close to the diagonal = good predictions.  
Systematic curves or fans = the model is missing a non-linear relationship.

In [ ]:
plt.figure(figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

# We need the global min/max across all models to draw the reference diagonal
all_preds = []
fitted_pipes = {}
for name, reg in models.items():
    pipe = Pipeline([("pre", preprocessor), ("reg", reg)])
    pipe.fit(X_train, y_train)
    fitted_pipes[name] = pipe
    all_preds.append(pipe.predict(X_test))

lo = min(float(np.min(y_test)), float(np.min([p.min() for p in all_preds])))
hi = max(float(np.max(y_test)), float(np.max([p.max() for p in all_preds])))

# Reference line — perfect prediction
plt.plot([lo, hi], [lo, hi], "k--", lw=3, label="Perfect Prediction", zorder=5)

# One scatter series per model
for (name, _), color, preds in zip(models.items(), colors, all_preds):
    r2 = r2_score(y_test, preds)
    plt.scatter(y_test, preds, alpha=0.55, s=35, color=color,
                label=f"{name} (R²={r2:.3f})")

plt.xlabel("Actual Values", fontsize=12)
plt.ylabel("Predicted Values", fontsize=12)
plt.title("Model Comparison: Predicted vs Actual (Test Set)",
          fontweight="bold", fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 14 — SHAP: Model Interpretability

Feature importance (Step 11) tells you *how much* each feature contributed on average.  
**SHAP** tells you *how* each feature affects each individual prediction — and in which direction.

**Three key plots:**

| Plot | What it shows |
|------|---------------|
| **Beeswarm** | Global view — which features matter most and do high values push the prediction up or down? |
| **Waterfall** | Local view — how was one specific prediction built up from the baseline? |
| **Dependence** | How does one feature's SHAP value change across its range? Are there thresholds or interactions? |

**Reading SHAP values:**
- **Positive SHAP** → feature pushed prediction *higher*
- **Negative SHAP** → feature pushed prediction *lower*
- `E[f(X)]` = average prediction across all training samples (the "baseline")
- `f(x)` = actual prediction for this sample = baseline + sum of all SHAP values

> 💡 Install with: `pip install shap`

In [ ]:
# ← UPDATE: Uncomment all cells in this section after running: pip install shap

# import shap
#
# # Step 1: Transform the data using the fitted preprocessor
# X_test_transformed  = best_pipeline.named_steps["pre"].transform(X_test)
# X_train_transformed = best_pipeline.named_steps["pre"].transform(X_train)
#
# # Step 2: Recover clean feature names (strip pipeline prefixes like 'num__', 'cat__')
# try:
#     raw_names   = best_pipeline.named_steps["pre"].get_feature_names_out()
#     clean_names = [n.split("__")[-1] for n in raw_names]
# except AttributeError:
#     clean_names = [f"Feature_{i}" for i in range(X_train_transformed.shape[1])]
#
# X_test_display  = pd.DataFrame(X_test_transformed,  columns=clean_names)
# X_train_display = pd.DataFrame(X_train_transformed, columns=clean_names)
#
# # Step 3: Create explainer — use a sample of training data as the "background" for speed
# background  = shap.sample(X_train_display, 100, random_state=RANDOM_STATE)
# explainer   = shap.Explainer(best_pipeline.named_steps["reg"], background)
# shap_values = explainer(X_test_display)
#
# print("✅ SHAP explainer ready.")

print("ℹ️  SHAP cells are commented out. Uncomment after: pip install shap")

In [ ]:
# SHAP Plot 1 — Beeswarm (global importance + direction)
# Each dot = one test prediction. x-axis = SHAP value. Colour = feature value (red=high, blue=low).
# Features sorted by mean |SHAP| — most important at the top.

# shap.plots.beeswarm(shap_values, max_display=12)

In [ ]:
# SHAP Plot 2 — Waterfall (single prediction broken down step by step)
# Shows exactly how each feature pushed the prediction up or down from the baseline.
# Change sample_idx to inspect a different row.

# sample_idx = 0
# shap.plots.waterfall(shap_values[sample_idx], max_display=12)

In [ ]:
# SHAP Plot 3 — Dependence (how does SHAP value vary with feature value?)
# Reveals non-linear thresholds. The colour shows the interacting feature ('auto' = strongest).

# top_feature = clean_names[0]   # ← UPDATE to the feature of interest
# shap.dependence_plot(top_feature, shap_values.values, X_test_display,
#                      interaction_index="auto")

---
## Step 15 — Conclusion & Checklist

Replace ⬜ with ✅ as you complete each step.

| Step | Status |
|------|--------|
| Loaded and inspected the dataset | ⬜ |
| Engineered new features (if applicable) | ⬜ |
| Ran EDA: target distribution, missing values, feature distributions, correlations | ⬜ |
| Split data — 80/20, split **before** any preprocessing | ⬜ |
| Built preprocessing pipeline (impute → collapse rare → encode → scale) | ⬜ |
| Compared 8 regressors using 5-fold cross-validation | ⬜ |
| Tuned hyperparameters on top models with GridSearchCV | ⬜ |
| Evaluated best model on held-out test set (R², MAE, RMSE, MAPE) | ⬜ |
| Plotted Predicted vs Actual and Residual plots | ⬜ |
| Analysed feature importance | ⬜ |
| Explained predictions using SHAP | ⬜ |

---

### Common pitfalls — check before submitting

| Pitfall | How to check |
|---------|--------------|
| **Data leakage** | Did you fit any preprocessor before splitting? |
| **Reporting CV score as final score** | Did you evaluate the final model on truly held-out test data? |
| **Forgetting to inverse-transform** | If you log-transformed the target, did you `np.expm1()` the predictions? |
| **Wrong metric for business goal** | Is R² the right success metric, or should it be MAE in dollars? |
| **Overfitting** | Is the gap between CV score and test score suspiciously large? |

---

### What to explore next

- **Larger search spaces:** `RandomizedSearchCV` instead of `GridSearchCV` for many parameters
- **Feature selection:** `RFECV` (recursive elimination with CV)
- **Stacking:** use base model predictions as features for a meta-model
- **Deployment:** `joblib.dump(best_pipeline, 'model.pkl')` → load and serve via API
- **Outlier handling:** `HuberRegressor` or `QuantileRegressor` for robustness to outliers
- **Time series:** if your data has a time dimension, use `TimeSeriesSplit` instead of `KFold`

In [ ]:
# Final performance summary — clean printout of all key metrics
final_metrics = {
    "Best Model": best_model_name,
    "R²":         round(r2,   4),
    "MAE":        round(mae,  4),
    "RMSE":       round(rmse, 4),
    "MAPE":       f"{mape:.2%}",
}

summary_df = pd.DataFrame(list(final_metrics.items()), columns=["Metric", "Value"])
print("\n" + "=" * 40)
print("  FINAL PERFORMANCE SUMMARY")
print("=" * 40)
print(summary_df.to_string(index=False))
print("\n🏆 Regression workflow complete!")
print("💡 To use your own data: replace the dummy dataset in Step 2 and set TARGET_COL in Step 4.")